# 🏦 Cashflow TFT Training Pipeline

**Stack:** PyTorch Forecasting · DagHub MLflow · Google Colab T4

**What this notebook does:**
1. Mounts Google Drive for persistent storage
2. Installs dependencies and clones your repo
3. Connects to DagHub as the free MLflow tracking server
4. Builds the PyTorch Forecasting `TimeSeriesDataSet` from your feature store
5. Trains a TFT model with static covariates (city, age, employment)
6. Evaluates with walk-forward CV — MAPE, RMSE, CI coverage
7. Logs everything to DagHub MLflow
8. Saves the trained model to Google Drive

---
**Before running:** Fill in the `CONFIG` cell (Cell 4) with your DagHub credentials and repo details.

## Cell 1 — Mount Google Drive
Run this first every session. All models, data, and logs persist here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Persistent directory layout on Drive ──────────────────────────────────────
BASE_DIR    = '/content/drive/MyDrive/cashflow_pipeline'
MODEL_DIR   = f'{BASE_DIR}/models'
DATA_DIR    = f'{BASE_DIR}/data'
LOG_DIR     = f'{BASE_DIR}/logs'
COHORT_DIR  = f'{BASE_DIR}/cohort_priors'
ARTIFACT_DIR= f'{BASE_DIR}/artifacts'

for d in [MODEL_DIR, DATA_DIR, LOG_DIR, COHORT_DIR, ARTIFACT_DIR]:
    os.makedirs(d, exist_ok=True)

print('✅ Drive mounted')
print(f'   Base : {BASE_DIR}')
print(f'   Models: {MODEL_DIR}')
print(f'   Data  : {DATA_DIR}')

## Cell 2 — Install Dependencies
Runs every session (~4 mins). Output suppressed — check the ✅ at the end.

In [ ]:
%%capture install_output

# Core ML
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install pytorch-forecasting pytorch-lightning

# MLflow + DagHub
!pip install mlflow dagshub

# Data + utils
!pip install pandas numpy scikit-learn xgboost shap optuna
!pip install prophet  # kept as cold-start fallback

print('✅ All dependencies installed')

In [ ]:
# Verify GPU is available
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Performance settings for Colab T4
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\n→ Training on: {DEVICE}')

## Cell 3 — Clone Your Repo
Pulls latest code from GitHub every session.

In [ ]:
import subprocess, sys

GITHUB_REPO = 'https://github.com/YOUR_USERNAME/cashflow_pipeline.git'  # ← change this
REPO_DIR    = '/content/cashflow_pipeline'

if os.path.exists(REPO_DIR):
    result = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print(f'Pulled latest: {result.stdout.strip()}')
else:
    result = subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], capture_output=True, text=True)
    print(f'Cloned: {result.stdout.strip() or result.stderr.strip()}')

# Add repo to Python path
sys.path.insert(0, REPO_DIR)
print(f'✅ Repo ready at {REPO_DIR}')

## Cell 4 — CONFIG
⚠️ **Fill these in before running anything else.**

DagHub setup:
1. Go to [dagshub.com](https://dagshub.com) → create free account
2. Create a new repo (or connect your GitHub repo)
3. Go to repo → Remote → Experiments → copy your MLflow tracking URL
4. Go to User Settings → Tokens → generate a token

In [ ]:
# ════════════════════════════════════════════════════════
#  FILL THESE IN
# ════════════════════════════════════════════════════════

DAGSHUB_USERNAME    = 'your_dagshub_username'       # your DagHub username
DAGSHUB_REPO_NAME   = 'cashflow_pipeline'           # your DagHub repo name
DAGSHUB_TOKEN       = 'your_dagshub_token'          # from DagHub → Settings → Tokens

# ════════════════════════════════════════════════════════
#  TRAINING CONFIG — tweak as needed
# ════════════════════════════════════════════════════════

CFG = {
    # Data
    'transactions_csv'  : f'{DATA_DIR}/sample_aa_transactions.csv',
    'min_history_months': 6,

    # TFT architecture
    'max_encoder_length' : 12,   # how many past months TFT looks at
    'max_prediction_length': 6,  # forecast horizon during training
    'hidden_size'        : 32,   # keep small for laptop/Colab Free
    'attention_head_size': 2,
    'dropout'            : 0.1,
    'hidden_continuous_size': 16,

    # Training
    'batch_size'    : 32,
    'max_epochs'    : 50,
    'learning_rate' : 3e-3,
    'gradient_clip' : 0.1,

    # Residual XGBoost
    'xgb_max_depth'   : 3,
    'xgb_n_estimators': 100,
    'min_months_for_residual': 6,  # need at least this many months to fit residual

    # MLflow
    'experiment_name': 'cashflow_tft_v1',
}

print('✅ Config set')
print(f'   Encoder length    : {CFG["max_encoder_length"]} months')
print(f'   Prediction length : {CFG["max_prediction_length"]} months')
print(f'   Hidden size       : {CFG["hidden_size"]}')
print(f'   Batch size        : {CFG["batch_size"]}')
print(f'   Max epochs        : {CFG["max_epochs"]}')

## Cell 5 — Connect to DagHub MLflow
This is your free remote tracking server. All runs visible at `dagshub.com/YOUR_USERNAME/cashflow_pipeline`.

In [ ]:
import mlflow
import dagshub

# Authenticate with DagHub
dagshub.init(
    repo_owner=DAGSHUB_USERNAME,
    repo_name=DAGSHUB_REPO_NAME,
    mlflow=True,
)

# Set token for auth (avoids interactive prompt)
os.environ['DAGSHUB_USER_TOKEN'] = DAGSHUB_TOKEN

# Verify connection
MLFLOW_TRACKING_URI = f'https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO_NAME}.mlflow'
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(CFG['experiment_name'])

print(f'✅ MLflow connected to DagHub')
print(f'   Tracking URI : {MLFLOW_TRACKING_URI}')
print(f'   Experiment   : {CFG["experiment_name"]}')
print(f'   View runs at : https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO_NAME}')

## Cell 6 — Load Data + Feature Engineering
Runs your existing pipeline modules to build the monthly feature store.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from src.ingestor import load_transactions
from src.transaction_categorizer import categorize_transactions
from src.feature_engineer import build_monthly_features

# ── Load raw transactions ─────────────────────────────────────────────────────
raw_df = load_transactions(CFG['transactions_csv'])
print(f'Raw transactions : {len(raw_df):,} rows')
print(f'Entities         : {raw_df["entity_id"].nunique()}')
print(f'Date range       : {raw_df["date"].min().date()} → {raw_df["date"].max().date()}')

# ── Categorize ───────────────────────────────────────────────────────────────
cat_df = categorize_transactions(raw_df)
print(f'\nCategory distribution:')
print(cat_df['category'].value_counts().head(10).to_string())

# ── Build feature store for all entities ─────────────────────────────────────
all_features = []
skipped = []

for entity_id in raw_df['entity_id'].unique():
    try:
        feats = build_monthly_features(cat_df, entity_id=entity_id)
        if len(feats) >= CFG['min_history_months']:
            all_features.append(feats)
        else:
            skipped.append((entity_id, len(feats), 'insufficient history'))
    except Exception as e:
        skipped.append((entity_id, 0, str(e)))

feature_store = pd.concat(all_features, ignore_index=True)
feature_store = feature_store.sort_values(['entity_id', 'period']).reset_index(drop=True)

# Add integer time index per entity (required by PyTorch Forecasting)
feature_store['time_idx'] = (
    feature_store.groupby('entity_id')['period']
    .transform(lambda s: (s - s.min()).dt.days // 30)
    .astype(int)
)

print(f'\n✅ Feature store built')
print(f'   Entities in store : {feature_store["entity_id"].nunique()}')
print(f'   Total rows        : {len(feature_store):,}')
print(f'   Skipped entities  : {len(skipped)}')
if skipped:
    for eid, n, reason in skipped[:5]:
        print(f'     {eid}: {reason} ({n} months)')

# Save to Drive for reuse
feature_store_path = f'{DATA_DIR}/feature_store.parquet'
feature_store.to_parquet(feature_store_path, index=False)
print(f'\n   Saved to: {feature_store_path}')

## Cell 7 — Add Static + Known Future Features
Enriches the feature store with user profile data and calendar signals that TFT uses as covariates.

In [ ]:
# ── Known future features (calendar signals TFT can see into the future) ──────
# These are things you KNOW in advance for future months

FESTIVAL_MONTHS = {10, 11}   # Oct-Nov: Diwali, Navratri
GST_FILING_MONTHS = {1, 4, 7, 10}   # Quarterly GST filing
ADVANCE_TAX_MONTHS = {3, 6, 9, 12}  # Advance tax due dates

feature_store['is_festival_month']    = feature_store['month'].isin(FESTIVAL_MONTHS).astype(int)
feature_store['is_gst_filing_month']  = feature_store['month'].isin(GST_FILING_MONTHS).astype(int)
feature_store['is_advance_tax_month'] = feature_store['month'].isin(ADVANCE_TAX_MONTHS).astype(int)
feature_store['quarter']              = ((feature_store['month'] - 1) // 3 + 1).astype(str)

# ── Static features (if you have a user profile CSV, load and merge it here) ──
# Expected schema: entity_id, city_tier, age_band, employment_type,
#                  household_size, cost_of_living_index

profile_path = f'{DATA_DIR}/user_profiles.csv'
if os.path.exists(profile_path):
    profiles = pd.read_csv(profile_path)
    feature_store = feature_store.merge(profiles, on='entity_id', how='left')
    print(f'✅ User profiles merged: {len(profiles)} profiles')
else:
    # Defaults when profile data isn't available yet
    print('⚠️  No user_profiles.csv found — using defaults')
    print('   Create data/user_profiles.csv with columns:')
    print('   entity_id, city_tier, age_band, employment_type, household_size')
    feature_store['city_tier']            = '2'         # default tier 2
    feature_store['age_band']             = 'growth'    # default 25-35
    feature_store['employment_type']      = 'salaried'  # default
    feature_store['household_size']       = 2
    feature_store['cost_of_living_index'] = 0.72

# Ensure categorical columns are strings
for col in ['city_tier', 'age_band', 'employment_type', 'quarter', 'entity_id']:
    feature_store[col] = feature_store[col].astype(str)

# Fill any remaining NaNs in numeric columns
NUMERIC_COLS = [
    'total_inflow', 'total_outflow', 'net_cashflow',
    'emi_to_inflow_ratio', 'fixed_obligation_ratio', 'savings_rate',
    'discretionary_spend_ratio', 'upi_to_inflow_ratio',
    'net_cashflow_lag1', 'net_cashflow_lag2', 'net_cashflow_lag3',
    'net_cashflow_roll3', 'net_cashflow_roll6',
    'inflow_mom_change', 'upi_spend_mom', 'fixed_obligation_mom',
    'cost_of_living_index', 'household_size',
]
for col in NUMERIC_COLS:
    if col in feature_store.columns:
        feature_store[col] = feature_store[col].fillna(0).astype(float)

print(f'\n✅ Feature enrichment complete')
print(f'   Columns: {list(feature_store.columns)}')
print(f'   Shape  : {feature_store.shape}')

## Cell 8 — Train / Val Split + PyTorch Forecasting Dataset
Builds the `TimeSeriesDataSet` that TFT expects. Chronological split — no leakage.

In [ ]:
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

# ── Chronological split ───────────────────────────────────────────────────────
# Use the last max_prediction_length months of each entity as validation
max_time_idx = feature_store.groupby('entity_id')['time_idx'].transform('max')
val_cutoff   = max_time_idx - CFG['max_prediction_length']

train_df = feature_store[feature_store['time_idx'] <= val_cutoff].copy()
val_df   = feature_store.copy()  # full data — dataset trims internally

print(f'Train rows : {len(train_df):,}')
print(f'Val rows   : {len(val_df):,}')

# ── Define feature groups for TFT ────────────────────────────────────────────
# TFT needs to know which features it can see into the future vs only the past

TIME_VARYING_KNOWN_REALS = [
    # Calendar — you always know the future month
    'month',
    'is_festival_month',
    'is_gst_filing_month',
    'is_advance_tax_month',
]

TIME_VARYING_UNKNOWN_REALS = [
    # Observed in the past, unknown in the future
    'net_cashflow',
    'total_inflow',
    'total_outflow',
    'emi_to_inflow_ratio',
    'fixed_obligation_ratio',
    'savings_rate',
    'discretionary_spend_ratio',
    'upi_to_inflow_ratio',
    'net_cashflow_lag1',
    'net_cashflow_lag2',
    'net_cashflow_lag3',
    'net_cashflow_roll3',
    'net_cashflow_roll6',
    'inflow_mom_change',
    'upi_spend_mom',
    'fixed_obligation_mom',
]

STATIC_CATEGORICALS = [
    'employment_type',
    'city_tier',
    'age_band',
]

STATIC_REALS = [
    'household_size',
    'cost_of_living_index',
]

TIME_VARYING_KNOWN_CATEGORICALS = ['quarter']

# Filter to columns that actually exist
def _filter_existing(cols):
    return [c for c in cols if c in feature_store.columns]

TVU_REALS = _filter_existing(TIME_VARYING_UNKNOWN_REALS)
TVK_REALS = _filter_existing(TIME_VARYING_KNOWN_REALS)
S_CATS    = _filter_existing(STATIC_CATEGORICALS)
S_REALS   = _filter_existing(STATIC_REALS)
TVK_CATS  = _filter_existing(TIME_VARYING_KNOWN_CATEGORICALS)

# ── Build TimeSeriesDataSet ───────────────────────────────────────────────────
training_dataset = TimeSeriesDataSet(
    train_df,
    time_idx                  = 'time_idx',
    target                    = 'net_cashflow',
    group_ids                 = ['entity_id'],          # one series per user
    max_encoder_length        = CFG['max_encoder_length'],
    max_prediction_length     = CFG['max_prediction_length'],
    static_categoricals       = S_CATS,
    static_reals              = S_REALS,
    time_varying_known_reals  = TVK_REALS,
    time_varying_known_categoricals = TVK_CATS,
    time_varying_unknown_reals= TVU_REALS,
    target_normalizer         = GroupNormalizer(
        groups=['entity_id'],
        transformation='softplus',   # handles negative cashflow gracefully
    ),
    add_relative_time_idx     = True,   # adds position encoding
    add_target_scales         = True,   # adds mean/std of target as features
    add_encoder_length        = True,   # tells model how much history is available
    allow_missing_timesteps   = True,   # handles cold-start users with gaps
)

# Validation dataset — same params, applied to full data
validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    val_df,
    predict=True,
    stop_randomization=True,
)

# DataLoaders
train_loader = training_dataset.to_dataloader(
    train=True,
    batch_size=CFG['batch_size'],
    num_workers=2,
    persistent_workers=True,
)
val_loader = validation_dataset.to_dataloader(
    train=False,
    batch_size=CFG['batch_size'] * 2,
    num_workers=2,
    persistent_workers=True,
)

print(f'\n✅ Datasets built')
print(f'   Training samples   : {len(training_dataset)}')
print(f'   Validation samples : {len(validation_dataset)}')
print(f'   Static categoricals: {S_CATS}')
print(f'   Static reals       : {S_REALS}')
print(f'   Known future reals : {TVK_REALS}')
print(f'   Unknown past reals : {TVU_REALS}')

## Cell 9 — Define TFT Model
Architecture config. Kept small for Colab Free T4 — scales up by increasing `hidden_size`.

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.loggers import MLFlowLogger

# ── TFT Model ────────────────────────────────────────────────────────────────
tft = TemporalFusionTransformer.from_dataset(
    training_dataset,

    # Architecture
    hidden_size             = CFG['hidden_size'],
    attention_head_size     = CFG['attention_head_size'],
    dropout                 = CFG['dropout'],
    hidden_continuous_size  = CFG['hidden_continuous_size'],

    # Loss: quantile loss gives you P10, P50, P90 directly
    loss = QuantileLoss(quantiles=[0.1, 0.5, 0.9]),

    # Optimizer
    learning_rate           = CFG['learning_rate'],
    reduce_on_plateau_patience = 4,   # reduce LR if val loss stalls for 4 epochs

    # Logging
    log_interval            = 10,
    log_val_interval        = 1,
)

print(f'✅ TFT model defined')
print(f'   Parameters: {sum(p.numel() for p in tft.parameters()):,}')
print(f'   Hidden size: {CFG["hidden_size"]}')
print(f'   Output: P10 / P50 / P90 quantiles per month')

# ── Callbacks ────────────────────────────────────────────────────────────────
checkpoint_path = f'{MODEL_DIR}/tft_best'

callbacks = [
    EarlyStopping(
        monitor  = 'val_loss',
        patience = 8,           # stop if val loss doesn't improve for 8 epochs
        mode     = 'min',
        verbose  = True,
    ),
    LearningRateMonitor(logging_interval='epoch'),
    ModelCheckpoint(
        dirpath   = checkpoint_path,
        filename  = 'tft-{epoch:02d}-{val_loss:.4f}',
        monitor   = 'val_loss',
        mode      = 'min',
        save_top_k= 1,          # only keep best checkpoint
        verbose   = True,
    ),
]

# ── MLflow Logger for Lightning ───────────────────────────────────────────────
mlflow_logger = MLFlowLogger(
    experiment_name = CFG['experiment_name'],
    tracking_uri    = MLFLOW_TRACKING_URI,
    run_name        = f'tft_hidden{CFG["hidden_size"]}_ep{CFG["max_epochs"]}',
)

# ── Trainer ───────────────────────────────────────────────────────────────────
trainer = pl.Trainer(
    max_epochs          = CFG['max_epochs'],
    accelerator         = 'gpu' if torch.cuda.is_available() else 'cpu',
    devices             = 1,
    gradient_clip_val   = CFG['gradient_clip'],
    callbacks           = callbacks,
    logger              = mlflow_logger,
    enable_progress_bar = True,
    log_every_n_steps   = 5,
)

print(f'\n✅ Trainer configured')
print(f'   Accelerator : {trainer.accelerator}')
print(f'   Max epochs  : {CFG["max_epochs"]} (early stopping at patience=8)')
print(f'   Checkpoints : {checkpoint_path}')

## Cell 10 — Train
This is where the GPU earns its keep. Expected time on T4: ~3–8 mins for 50 epochs at beta scale.

In [ ]:
import time

print('🚀 Starting TFT training...')
print(f'   Logging to DagHub: https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO_NAME}')
print('─' * 60)

t0 = time.time()
trainer.fit(
    tft,
    train_dataloaders = train_loader,
    val_dataloaders   = val_loader,
)
elapsed = time.time() - t0

print('─' * 60)
print(f'✅ Training complete in {elapsed/60:.1f} mins')
print(f'   Best epoch     : {trainer.current_epoch}')
print(f'   Best val loss  : {trainer.checkpoint_callback.best_model_score:.4f}')
print(f'   Checkpoint at  : {trainer.checkpoint_callback.best_model_path}')

# Log training time to MLflow
with mlflow.start_run(run_id=mlflow_logger.run_id):
    mlflow.log_metric('training_time_minutes', elapsed / 60)
    mlflow.log_params(CFG)

## Cell 11 — Load Best Checkpoint + Evaluate
Loads the best checkpoint (lowest val loss) and computes MAPE, RMSE, and CI coverage.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Load best model ───────────────────────────────────────────────────────────
best_model = TemporalFusionTransformer.load_from_checkpoint(
    trainer.checkpoint_callback.best_model_path
)
best_model.eval()
print(f'✅ Best model loaded')

# ── Generate predictions on validation set ────────────────────────────────────
predictions = best_model.predict(
    val_loader,
    mode          = 'quantiles',   # returns P10, P50, P90
    return_index  = True,
    return_decoder_lengths = True,
)

# ── Extract actuals and predictions ──────────────────────────────────────────
actuals_raw, pred_raw = best_model.predict(
    val_loader,
    mode         = 'raw',
    return_x     = True,
)

# P10, P50 (point forecast), P90 from quantile output
pred_quantiles = predictions.output  # shape: (n_samples, prediction_length, 3)
p10 = pred_quantiles[..., 0].numpy().flatten()
p50 = pred_quantiles[..., 1].numpy().flatten()   # point forecast
p90 = pred_quantiles[..., 2].numpy().flatten()

# Actuals
actuals_list = []
for x, _ in val_loader:
    actuals_list.append(x['decoder_target'].numpy())
actuals = np.concatenate(actuals_list).flatten()

# Align lengths
n = min(len(actuals), len(p50))
actuals, p10, p50, p90 = actuals[:n], p10[:n], p50[:n], p90[:n]

# ── Compute metrics ───────────────────────────────────────────────────────────
def mape(a, p):
    mask = a != 0
    return float(np.mean(np.abs((a[mask] - p[mask]) / a[mask])) * 100)

def coverage(a, lo, hi):
    return float(np.mean((a >= lo) & (a <= hi)) * 100)

metrics = {
    'val_mape'       : mape(actuals, p50),
    'val_rmse'       : float(np.sqrt(mean_squared_error(actuals, p50))),
    'val_mae'        : float(mean_absolute_error(actuals, p50)),
    'val_r2'         : float(r2_score(actuals, p50)),
    'val_ci_coverage': coverage(actuals, p10, p90),
}

print('\n── Validation Metrics ──────────────────────────────────')
print(f'  MAPE         : {metrics["val_mape"]:.1f}%   (lower is better)')
print(f'  RMSE         : {metrics["val_rmse"]:,.0f}')
print(f'  MAE          : {metrics["val_mae"]:,.0f}')
print(f'  R²           : {metrics["val_r2"]:+.3f}   (1.0 = perfect)')
print(f'  CI Coverage  : {metrics["val_ci_coverage"]:.1f}%  (target: >70%)')
print('──────────────────────────────────────────────────────────')

# Log to MLflow
with mlflow.start_run(run_id=mlflow_logger.run_id):
    mlflow.log_metrics(metrics)

print(f'\n✅ Metrics logged to DagHub')

## Cell 12 — Attention + Variable Importance
TFT's built-in interpretability — see which features and which past months it's attending to.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# ── Variable importance (which features does TFT rely on most?) ───────────────
interpretation = best_model.interpret_output(
    best_model.predict(val_loader, mode='raw', return_x=True),
    reduction='sum',
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('TFT Variable Importance', fontsize=13, fontweight='bold')

# Encoder variable importance (past features)
enc_importance = interpretation['encoder_variables']
enc_df = pd.DataFrame({
    'feature': list(enc_importance.keys()),
    'importance': list(enc_importance.values())
}).sort_values('importance', ascending=True)

axes[0].barh(enc_df['feature'], enc_df['importance'], color='#2563eb', alpha=0.8)
axes[0].set_title('Past (Encoder) Variables')
axes[0].set_xlabel('Importance')

# Decoder variable importance (future features)
dec_importance = interpretation['decoder_variables']
dec_df = pd.DataFrame({
    'feature': list(dec_importance.keys()),
    'importance': list(dec_importance.values())
}).sort_values('importance', ascending=True)

axes[1].barh(dec_df['feature'], dec_df['importance'], color='#16a34a', alpha=0.8)
axes[1].set_title('Future (Decoder) Variables')
axes[1].set_xlabel('Importance')

plt.tight_layout()

# Save to Drive + log to MLflow
vi_path = f'{ARTIFACT_DIR}/variable_importance.png'
plt.savefig(vi_path, bbox_inches='tight')
plt.show()

with mlflow.start_run(run_id=mlflow_logger.run_id):
    mlflow.log_artifact(vi_path)

print(f'\n✅ Variable importance saved to {vi_path}')
print('\nTop 3 encoder variables (what TFT relies on most from the past):')
print(enc_df.tail(3)[['feature', 'importance']].to_string(index=False))

## Cell 13 — XGBoost Personal Residual Layer
Trains a per-user XGBoost on TFT's residuals. Corrects systematic errors for individual users.

In [ ]:
import xgboost as xgb
import joblib

RESIDUAL_MODEL_DIR = f'{MODEL_DIR}/residual_models'
os.makedirs(RESIDUAL_MODEL_DIR, exist_ok=True)

RESIDUAL_FEATURES = [
    'net_cashflow_lag1', 'net_cashflow_lag2', 'net_cashflow_lag3',
    'net_cashflow_roll3', 'net_cashflow_roll6',
    'emi_to_inflow_ratio', 'savings_rate', 'fixed_obligation_ratio',
    'inflow_mom_change', 'month',
]

residual_models = {}
residual_report = []

for entity_id in feature_store['entity_id'].unique():
    entity_df = feature_store[feature_store['entity_id'] == entity_id].copy()
    entity_df = entity_df.sort_values('period').reset_index(drop=True)
    n = len(entity_df)

    if n < CFG['min_months_for_residual']:
        continue   # not enough history for personal residual

    # Get TFT predictions for this entity's history
    # (in production you'd run TFT inference here; for now we approximate
    #  with the validation predictions we already have)
    # Residual = actual - TFT_predicted
    # We use net_cashflow_roll3 as a proxy TFT prediction for scaffold purposes
    entity_df['tft_pred_proxy'] = entity_df['net_cashflow_roll3'].shift(1).fillna(0)
    entity_df['residual']       = entity_df['net_cashflow'] - entity_df['tft_pred_proxy']

    # Features and target
    avail_features = [f for f in RESIDUAL_FEATURES if f in entity_df.columns]
    X = entity_df[avail_features].fillna(0)
    y = entity_df['residual'].fillna(0)

    # Train/test split (last 2 months held out)
    X_train, X_test = X.iloc[:-2], X.iloc[-2:]
    y_train, y_test = y.iloc[:-2], y.iloc[-2:]

    if len(X_train) < 3:
        continue

    model = xgb.XGBRegressor(
        n_estimators  = CFG['xgb_n_estimators'],
        max_depth     = CFG['xgb_max_depth'],
        learning_rate = 0.05,
        subsample     = 0.8,
        reg_lambda    = 1.0,
        reg_alpha     = 0.1,
        random_state  = 42,
        verbosity     = 0,
    )
    model.fit(X_train, y_train)

    # Score on held-out months
    if len(X_test) > 0:
        residual_rmse = float(np.sqrt(mean_squared_error(y_test, model.predict(X_test))))
        residual_report.append({'entity_id': entity_id, 'residual_rmse': residual_rmse, 'n_months': n})

    # Save model
    model_path = f'{RESIDUAL_MODEL_DIR}/residual_{entity_id}.joblib'
    joblib.dump(model, model_path)
    residual_models[entity_id] = model

print(f'✅ Residual models trained: {len(residual_models)} users')
if residual_report:
    report_df = pd.DataFrame(residual_report)
    print(f'   Avg residual RMSE : {report_df["residual_rmse"].mean():,.0f}')
    print(f'   Avg history length: {report_df["n_months"].mean():.1f} months')

    # Log aggregate residual metrics
    with mlflow.start_run(run_id=mlflow_logger.run_id):
        mlflow.log_metric('residual_avg_rmse', report_df['residual_rmse'].mean())
        mlflow.log_metric('residual_n_users', len(residual_models))

## Cell 14 — Hierarchical Blending + Final Inference
Combines TFT global forecast + XGBoost residual correction using history-length-based weights.

In [ ]:
def get_blend_weights(n_months: int) -> dict:
    """Returns blend weights based on how much history the user has."""
    if n_months < 3:
        return {'tft': 0.00, 'residual': 0.00, 'cohort': 1.00}
    elif n_months < 6:
        return {'tft': 0.30, 'residual': 0.10, 'cohort': 0.60}
    elif n_months < 12:
        return {'tft': 0.65, 'residual': 0.35, 'cohort': 0.00}
    elif n_months < 24:
        return {'tft': 0.45, 'residual': 0.55, 'cohort': 0.00}
    else:
        return {'tft': 0.30, 'residual': 0.70, 'cohort': 0.00}


def hierarchical_predict(
    entity_id: str,
    feature_store: pd.DataFrame,
    tft_model,
    residual_models: dict,
    horizon: int = 6,
) -> pd.DataFrame:
    """
    Run hierarchical prediction for one user:
      final = w_tft * tft_forecast + w_residual * residual_correction

    Returns DataFrame with columns:
      period, tft_forecast, residual_correction, final_forecast, p10, p90, weights
    """
    entity_df = feature_store[feature_store['entity_id'] == entity_id].copy()
    n_months  = len(entity_df)
    weights   = get_blend_weights(n_months)

    last_period = entity_df['period'].max()
    future_periods = pd.date_range(
        start=last_period + pd.offsets.MonthBegin(1),
        periods=horizon, freq='MS'
    )

    # ── TFT forecast (returns P10, P50, P90) ─────────────────────────────────
    # In production: run tft_model.predict() on entity's TimeSeriesDataSet
    # Here we use a placeholder — replace with real TFT inference in production
    last_cf     = float(entity_df['net_cashflow'].iloc[-1])
    trend       = float(entity_df['net_cashflow'].diff().mean())
    tft_p50     = np.array([last_cf + trend * (i + 1) for i in range(horizon)])
    residual_std = float(entity_df['net_cashflow'].std()) * 1.28
    tft_p10     = tft_p50 - residual_std
    tft_p90     = tft_p50 + residual_std

    # ── XGBoost residual correction ───────────────────────────────────────────
    residual_correction = np.zeros(horizon)
    if entity_id in residual_models and weights['residual'] > 0:
        avail_features = [f for f in RESIDUAL_FEATURES if f in entity_df.columns]
        last_row = entity_df[avail_features].fillna(0).iloc[[-1]]
        # Apply same last-row features for all horizon steps (simplification)
        # In production: update lag features recursively per step
        for i in range(horizon):
            residual_correction[i] = residual_models[entity_id].predict(last_row)[0]

    # ── Blend ─────────────────────────────────────────────────────────────────
    final_forecast = (
        weights['tft']      * tft_p50 +
        weights['residual'] * (tft_p50 + residual_correction)
    )

    return pd.DataFrame({
        'period'              : future_periods,
        'tft_forecast'        : tft_p50,
        'residual_correction' : residual_correction,
        'final_forecast'      : final_forecast,
        'p10'                 : tft_p10,
        'p90'                 : tft_p90,
        'w_tft'               : weights['tft'],
        'w_residual'          : weights['residual'],
        'n_months_history'    : n_months,
    })


# ── Demo: run for first entity ────────────────────────────────────────────────
sample_entity = feature_store['entity_id'].iloc[0]
n_months = len(feature_store[feature_store['entity_id'] == sample_entity])
weights = get_blend_weights(n_months)

result = hierarchical_predict(
    entity_id      = sample_entity,
    feature_store  = feature_store,
    tft_model      = best_model,
    residual_models= residual_models,
    horizon        = 6,
)

print(f'✅ Hierarchical forecast for entity: {sample_entity}')
print(f'   History length : {n_months} months')
print(f'   Blend weights  : TFT={weights["tft"]:.0%}  Residual={weights["residual"]:.0%}  Cohort={weights["cohort"]:.0%}')
print()
print(result[['period', 'tft_forecast', 'residual_correction', 'final_forecast', 'p10', 'p90']].to_string(index=False))

## Cell 15 — Save Everything to Drive + Log Final Artifacts

In [ ]:
import shutil
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M')

# ── Save TFT model ────────────────────────────────────────────────────────────
tft_save_path = f'{MODEL_DIR}/tft_model_{timestamp}.ckpt'
shutil.copy(trainer.checkpoint_callback.best_model_path, tft_save_path)
print(f'✅ TFT model saved: {tft_save_path}')

# ── Save training dataset params (needed to reconstruct dataset for inference) 
import pickle
dataset_params_path = f'{MODEL_DIR}/training_dataset_params_{timestamp}.pkl'
with open(dataset_params_path, 'wb') as f:
    pickle.dump(training_dataset.get_parameters(), f)
print(f'✅ Dataset params saved: {dataset_params_path}')

# ── Save feature store ────────────────────────────────────────────────────────
fs_save_path = f'{DATA_DIR}/feature_store_{timestamp}.parquet'
feature_store.to_parquet(fs_save_path, index=False)
print(f'✅ Feature store saved: {fs_save_path}')

# ── Log to MLflow ─────────────────────────────────────────────────────────────
with mlflow.start_run(run_id=mlflow_logger.run_id):
    mlflow.log_artifact(tft_save_path, artifact_path='model')
    mlflow.log_artifact(dataset_params_path, artifact_path='model')
    mlflow.log_param('model_timestamp', timestamp)
    mlflow.log_param('n_entities_trained', feature_store['entity_id'].nunique())
    mlflow.log_param('n_residual_models', len(residual_models))

print(f'\n✅ All artifacts logged to MLflow on DagHub')
print(f'   View at: https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO_NAME}')
print()
print('── Summary ─────────────────────────────────────────────────')
print(f'  TFT val MAPE     : {metrics["val_mape"]:.1f}%')
print(f'  TFT val RMSE     : {metrics["val_rmse"]:,.0f}')
print(f'  CI Coverage      : {metrics["val_ci_coverage"]:.1f}%')
print(f'  Residual models  : {len(residual_models)} users')
print(f'  MLflow run ID    : {mlflow_logger.run_id}')
print('────────────────────────────────────────────────────────────')

## Cell 16 — Reload Model (Next Session)
Run this instead of training to load a previously saved model from Drive.

In [ ]:
# ── Uncomment and run this to load a saved model without retraining ────────────

# import pickle, joblib, glob
#
# # Find latest model
# model_files = sorted(glob.glob(f'{MODEL_DIR}/tft_model_*.ckpt'))
# latest_model_path = model_files[-1]
#
# # Find matching dataset params
# ts = latest_model_path.split('tft_model_')[1].replace('.ckpt', '')
# dataset_params_path = f'{MODEL_DIR}/training_dataset_params_{ts}.pkl'
#
# # Rebuild dataset from saved params (needed for inference)
# with open(dataset_params_path, 'rb') as f:
#     dataset_params = pickle.load(f)
#
# # Load TFT
# best_model = TemporalFusionTransformer.load_from_checkpoint(latest_model_path)
# best_model.eval()
#
# # Load residual models
# residual_models = {}
# for path in glob.glob(f'{RESIDUAL_MODEL_DIR}/residual_*.joblib'):
#     entity_id = path.split('residual_')[1].replace('.joblib', '')
#     residual_models[entity_id] = joblib.load(path)
#
# print(f'✅ Loaded TFT from: {latest_model_path}')
# print(f'   Residual models : {len(residual_models)}')

print('Uncomment the block above to load a saved model from Drive.')